In [ ]:
from utilities import initialize_with_module

await initialize_with_module([
        'AITW_infiltration_surrogate/1.0.0-foss-2023a',
        'matplotlib/3.7.2-gfbf-2023a',
    ],
    '2023.06'
)

In [ ]:
import os

import numpy as np
from IPython.display import Image as IPyImage
import matplotlib.pyplot as plt

def loss_finder(base_path: str) -> list[tuple[str, str]]:
    """
    Find .dat files containing loss values in a base path.

    Parameters:
    base_path (str): The base directory to search for .dat files.

    Returns:
    list[tuple[str, str]]: A list of tuples containing the name of the directory and the path to the .dat file.
    """
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file == 'loss_curve.dat':
                yield os.path.basename(root), os.path.join(root, file)

def loss_loader(dat_file_path: str) -> tuple[np.ndarray, str]:
    """
    Load the loss values from a .dat file.

    Parameters:
    dat_file_path (str): Path to the .dat file containing loss values.

    Returns:
    np.ndarray: Array of loss values.
    str: Path to the .dat file.
    """
    try:
        # Load the data from the .dat file
        data = np.loadtxt(dat_file_path)
        return data, dat_file_path
    except Exception as e:
        print(f"Error loading loss values from {dat_file_path}: {e}")
        return None, None  # Return an empty array in case of error

def loss_visualizer(loss_data_path: tuple[np.ndarray, str]) -> None:
    """
    Visualize the loss values as a line plot.
    Column 1: Epochs
    Column 2: Training Loss
    Column 3: Validation Loss

    Parameters:
    loss_values (np.ndarray): Array of loss values to visualize.
    title (str): Title of the plot.
    xlabel (str): Label for the x-axis.
    ylabel (str): Label for the y-axis.
    """
    loss_data, loss_path = loss_data_path

    basedir = os.path.dirname(loss_path)

    # Create a line plot for the loss values
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(loss_data[:, 0], loss_data[:, 1], label='Training Loss', color='blue')
    ax.plot(loss_data[:, 0], loss_data[:, 2], label='Validation Loss', color='orange')
    # Y-logscale
    ax.set_yscale('log')
    ax.set_title('Training and Validation Loss Over Epochs')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Loss')
    ax.legend()

    # Save the plot as a PNG file in the same directory as the .dat file
    plot_path = os.path.join(basedir, 'loss_plot.png')
    plt.savefig(plot_path)

    image = IPyImage(filename=plot_path)
    display(image)


In [ ]:
import os
import utilities
from common import INFILTRATION_SURROGATE_INFERENCE_DEFAULTS, INFILTRATION_SURROGATE_TRAIN_DEFAULTS, INFILTRATION_SURROGATE_OUTDIR
import ipywidgets as wdg
from aitw_infiltration_surrogate.cli.inference import inference
from aitw_infiltration_surrogate.cli.train import train

train_outdir = os.path.join(INFILTRATION_SURROGATE_OUTDIR, "train")

widgets_infiltr, infiltr_event = utilities.click_cli_widget(
    inference,
    override_defaults=INFILTRATION_SURROGATE_INFERENCE_DEFAULTS,
    title="AITW - Surrogate Infiltration evaluation",
    description_text="",
    button_description="Evaluate Infiltration",
    button_running_text="Evaluating...",
    output_file_param="output_file"
)


visualizer = utilities.output_visualizer_widget(
    title="Wood Stiffness Results",
    file_format="csv",
    event_observer=infiltr_event,
    output_dir=INFILTRATION_SURROGATE_OUTDIR
)
visualizer_accordion = wdg.Accordion(children=[visualizer], selected_index=None)
visualizer_accordion.set_title(0, "Inference Results")

widgets_train, train_event = utilities.click_cli_widget(
    train,
    override_defaults=INFILTRATION_SURROGATE_TRAIN_DEFAULTS,
    title="AITW - Surrogate Infiltration model training",
    description_text="",
    button_description="Train Infiltration Model",
    button_running_text="Training...",
)
train_visualizer = utilities.output_visualizer_widget(
    title="Training Loss Results",
    # file_format="dat",
    event_observer=train_event,
    output_dir=train_outdir,
    outputs_func=lambda: loss_finder(INFILTRATION_SURROGATE_OUTDIR),
    read_func=loss_loader,
    data_display_func=loss_visualizer,
)
train_visualizer_accordion = wdg.Accordion(children=[train_visualizer], selected_index=None)
train_visualizer_accordion.set_title(0, "Training Loss Results")
train_box = wdg.VBox([widgets_train, train_visualizer_accordion])
train_accordion = wdg.Accordion(children=[train_box], selected_index=None)
train_accordion.set_title(0, "Train Surrogate Model")

main_vbox = wdg.VBox([widgets_infiltr, visualizer_accordion, train_accordion])
display(main_vbox)

In [ ]:
import ipywidgets as wdg
spacer = wdg.Box(
    layout=wdg.Layout(height="95vh", flex="0 0 auto")
)
display(spacer)